####Cell 1 — _Imports_

In [0]:
from pyspark.sql import functions as F


####Cell 2 — Table configuration

In [0]:
silver_table = (
     "workspace.nyc_taxi_aws.silver_taxi_trips_curated"
  )

gold_daily_table = (
    "workspace.nyc_taxi_aws.gold_daily_metrics_aws"
)

gold_hourly_table = (
    "workspace.nyc_taxi_aws.gold_hourly_metrics_aws"
)

gold_payment_table = (
    "workspace.nyc_taxi_aws.gold_payment_metrics_aws"
)

gold_vendor_table = (
    "workspace.nyc_taxi_aws.gold_vendor_metrics_aws"
)

gold_dashboard_metrics_aws=(
    "workspace.nyc_taxi_aws.gold_dashboard_metrics_aws"
)



#### Create Silver DataFrame

In [0]:
silver_df = spark.table(silver_table)

#### Creating Daily Metrics Table using silver dataFrame

In [0]:
#Daily Metrics

daily_metrics_df = (
    silver_df
    .groupBy(
        "pickup_date"
    )
    .agg(
        F.count("*").alias("total_trips"),

        F.round(
            F.avg("trip_distance"),
            2
        ).alias("avg_trip_distance"),

        F.round(
            F.avg("trip_duration_minutes"),
            2
        ).alias("avg_trip_duration_minutes"),

        F.round(
            F.avg("fare_amount"),
            2
        ).alias("avg_fare_amount"),

        F.round(
            F.sum("total_amount"),
            2
        ).alias("total_revenue"),

        F.round(
            F.avg("fare_per_mile"),
            2
        ).alias("avg_fare_per_mile")
    )
    .withColumn(
        "gold_updated_at",
        F.current_timestamp()
    )
)


(
    daily_metrics_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(gold_daily_table)
)

#### Creating Hourly Metrics Table using Silver dataframe

In [0]:
#Hourly metrics

hourly_metrics_df = (
    silver_df
    .groupBy(
        "pickup_hour"
    )
    .agg(
        F.count("*").alias("total_trips"),

        F.round(
            F.avg("trip_distance"),
            2
        ).alias("avg_trip_distance"),

        F.round(
            F.avg("trip_duration_minutes"),
            2
        ).alias("avg_trip_duration_minutes"),

        F.round(
            F.avg("total_amount"),
            2
        ).alias("avg_total_amount"),

        F.round(
            F.sum("total_amount"),
            2
        ).alias("total_revenue")
    )
    .orderBy(
        "pickup_hour"
    )
    .withColumn(
        "gold_updated_at",
        F.current_timestamp()
    )
)

(
    hourly_metrics_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(gold_hourly_table)
)

#### Creating Payment Metrics Table using Silver dataframe

In [0]:
#Payment Metrics
payment_metrics_df = (
    silver_df
    .groupBy(
        "payment_type"
    )
    .agg(
        F.count("*").alias("total_trips"),

        F.round(
            F.avg("fare_amount"),
            2
        ).alias("avg_fare_amount"),

        F.round(
            F.avg("total_amount"),
            2
        ).alias("avg_total_amount"),

        F.round(
            F.sum("total_amount"),
            2
        ).alias("total_revenue"),

        F.round(
            F.avg("trip_distance"),
            2
        ).alias("avg_trip_distance")
    )
    .withColumn(
        "payment_type_name",
        F.when(F.col("payment_type") == 0, "Unknown")
        .when(F.col("payment_type") == 1, "Credit Card")
        .when(F.col("payment_type") == 2, "Cash")
        .when(F.col("payment_type") == 3, "No Charge")
        .when(F.col("payment_type") == 4, "Dispute")
        .otherwise("Unknown")
    )
    .withColumn(
        "gold_updated_at",
        F.current_timestamp()
    )
)

(
    payment_metrics_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(gold_payment_table)
)

#### Creating Vendor Metrics Table using Silver dataframe

In [0]:
# Gold Table 4 — Vendor Analysis

vendor_metrics_df = (
    silver_df
    .groupBy(
        "VendorID"
    )
    .agg(
        F.count("*").alias("total_trips"),

        F.round(
            F.avg("trip_distance"),
            2
        ).alias("avg_trip_distance"),

        F.round(
            F.avg("trip_duration_minutes"),
            2
        ).alias("avg_trip_duration_minutes"),

        F.round(
            F.avg("fare_amount"),
            2
        ).alias("avg_fare_amount"),

        F.round(
            F.sum("total_amount"),
            2
        ).alias("total_revenue"),

        F.round(
            F.avg("total_amount"),
            2
        ).alias("avg_total_amount"),

        F.round(
            F.avg("passenger_count"),
            2
        ).alias("avg_passengers")
    )
    .withColumn(
        "vendor_name",
        F.when(F.col("VendorID") == 1, "Creative Mobile Technologies")
        .when(F.col("VendorID") == 2, "VeriFone Inc.")
        .otherwise("Unknown")
    )
    .withColumn(
        "gold_updated_at",
        F.current_timestamp()
    )
    .select(
        "VendorID",
        "vendor_name",
        "total_trips",
        "total_revenue",
        "avg_fare_amount",
        "avg_total_amount",
        "avg_trip_distance",
        "avg_trip_duration_minutes",
        "avg_passengers",
        "gold_updated_at"
    )
)

(
    vendor_metrics_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(gold_vendor_table)
)

####Creating Dashboard Metrics Table using Silver table

In [0]:

dashboard_metrics_df = (
    spark.table(
        silver_table
    )
    .withColumn(
        "pickup_date",
        F.to_date("pickup_datetime")
    )
    .withColumn(
        "time_of_day",
        F.when(
            F.col("pickup_hour").between(5, 11),
            "Morning"
        ).when(
            F.col("pickup_hour").between(12, 16),
            "Afternoon"
        ).when(
            F.col("pickup_hour").between(17, 21),
            "Evening"
        ).otherwise(
            "Night"
        )
    )
    .withColumn(
        "is_weekend",
        F.dayofweek("pickup_date").isin(1, 7)
    )
    .groupBy(
        "pickup_date"
    )
    .agg(

        F.count("*").alias(
            "total_trips"
        ),

        F.round(
            F.sum("total_amount"),
            2
        ).alias(
            "total_revenue"
        ),

        F.round(
            F.avg("fare_amount"),
            2
        ).alias(
            "average_fare"
        ),

        F.round(
            F.avg("trip_distance"),
            2
        ).alias(
            "average_trip_distance"
        ),

        F.round(
            F.avg("trip_duration_minutes"),
            2
        ).alias(
            "average_trip_duration"
        ),

        F.round(
            F.avg("average_speed_mph"),
            2
        ).alias(
            "average_speed"
        ),

        F.round(
            F.avg("tip_amount"),
            2
        ).alias(
            "average_tip"
        ),

        F.sum(
            F.when(
                F.col("payment_type") == 1,
                1
            ).otherwise(
                0
            )
        ).alias(
            "credit_card_trips"
        ),

        F.sum(
            F.when(
                F.col("payment_type") == 2,
                1
            ).otherwise(
                0
            )
        ).alias(
            "cash_trips"
        ),

        F.sum(
            F.when(
                F.col("VendorID") == 1,
                1
            ).otherwise(
                0
            )
        ).alias(
            "vendor1_trips"
        ),

        F.sum(
            F.when(
                F.col("VendorID") == 2,
                1
            ).otherwise(
                0
            )
        ).alias(
            "vendor2_trips"
        ),

        F.round(
            F.sum(
                F.when(
                    F.col("VendorID") == 1,
                    F.col("total_amount")
                ).otherwise(
                    0
                )
            ),
            2
        ).alias(
            "vendor1_revenue"
        ),

        F.round(
            F.sum(
                F.when(
                    F.col("VendorID") == 2,
                    F.col("total_amount")
                ).otherwise(
                    0
                )
            ),
            2
        ).alias(
            "vendor2_revenue"
        ),

        F.sum(
            F.when(
                F.col("PULocationID").isin(
                    1,
                    132,
                    138
                ),
                1
            ).otherwise(
                0
            )
        ).alias(
            "airport_pickups"
        ),

        F.sum(
            F.when(
                F.col("is_weekend"),
                1
            ).otherwise(
                0
            )
        ).alias(
            "weekend_trips"
        ),

        F.sum(
            F.when(
                ~F.col("is_weekend"),
                1
            ).otherwise(
                0
            )
        ).alias(
            "weekday_trips"
        ),

        F.sum(
            F.when(
                F.col("pickup_hour").between(5, 11),
                1
            ).otherwise(
                0
            )
        ).alias(
            "morning_trips"
        ),

        F.sum(
            F.when(
                F.col("pickup_hour").between(12, 16),
                1
            ).otherwise(
                0
            )
        ).alias(
            "afternoon_trips"
        ),

        F.sum(
            F.when(
                F.col("pickup_hour").between(17, 21),
                1
            ).otherwise(
                0
            )
        ).alias(
            "evening_trips"
        ),

        F.sum(
            F.when(
                (F.col("pickup_hour") >= 22)
                | (F.col("pickup_hour") <= 4),
                1
            ).otherwise(
                0
            )
        ).alias(
            "night_trips"
        )

    )
)

(
    dashboard_metrics_df.write
    .format(
        "delta"
    )
    .mode(
        "overwrite"
    )
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "workspace.nyc_taxi_aws.gold_dashboard_metrics_aws"
    )
)

display(
    dashboard_metrics_df
)

pickup_date,total_trips,total_revenue,average_fare,average_trip_distance,average_trip_duration,average_speed,average_tip,credit_card_trips,cash_trips,vendor1_trips,vendor2_trips,vendor1_revenue,vendor2_revenue,airport_pickups,weekend_trips,weekday_trips,morning_trips,afternoon_trips,evening_trips,night_trips
2026-04-07,115313,3395656.52,20.23,4.26,17.46,13.1,3.11,85900,11647,24890,90068,728381.29,2656193.9,8289,0,115313,29301,33188,38720,14104
2009-01-01,7,258.15,25.3,4.42,157.19,10.69,2.14,2,5,0,7,0.0,258.15,2,0,7,1,2,0,4
2026-05-23,113381,3151605.49,19.9,5.64,16.38,22.78,2.45,67727,11107,19506,93707,519894.99,2626257.74,5480,113381,0,17968,32529,35250,27634
2025-11-28,84139,2284212.08,17.98,5.97,16.06,20.78,3.12,60499,11718,17699,66373,467194.55,1814801.49,6899,0,84139,16642,29537,25852,12108
2026-05-05,123505,3735725.07,20.98,4.53,19.18,15.41,3.18,91521,11808,26686,96379,783424.45,2939004.06,7364,0,123505,31410,34738,41209,16148
2025-10-16,145448,4533052.6,21.7,6.7,21.15,20.11,3.46,106233,12796,31350,113977,935294.41,3593556.37,10506,0,145448,35632,38849,49508,21459
2025-10-01,122623,3563243.3,19.97,8.7,18.5,22.01,3.21,90284,11446,27618,94883,770246.01,2789095.92,7723,0,122623,33176,34842,39508,15097
2025-11-21,138647,4100037.74,20.61,8.14,19.43,27.15,3.09,97325,12114,31020,107438,888623.19,3205057.21,8826,0,138647,32600,37919,41563,26565
2026-04-04,120593,3330411.81,19.65,3.8,16.01,13.2,2.51,76820,12297,20941,99485,556827.61,2768089.61,5264,120593,0,18426,33170,36771,32226
2025-10-15,133434,3999091.77,20.55,4.49,19.02,15.5,3.39,99795,12249,28904,104403,836811.05,3158099.84,10023,0,133434,33278,36931,45404,17821


####Printing count of all the gold Tables records

In [0]:
print(
    "Gold tables successfully created."
)

print(
    f"Daily metrics: {daily_metrics_df.count()}"
)

print(
    f"Hourly metrics: {hourly_metrics_df.count()}"
)

print(
    f"Payment metrics: {payment_metrics_df.count()}"
)

print(
    f"Vendor metrics: {vendor_metrics_df.count()}"
)

print(
    f"Dashboard metrics: {dashboard_metrics_df.count()}"
)

Gold tables successfully created.
Daily metrics: 278
Hourly metrics: 24
Payment metrics: 5
Vendor metrics: 3
Dashboard metrics: 278
